# nb_<vrstva>_<akce>

| | |
|---|---|
| **Účel** | Jedna věta: co notebook dělá. |
| **Vrstva** | bronze → silver |
| **Vstup** | `bronze_<tabulka>` |
| **Výstup** | `silver_<tabulka>`, `ops_run_log` |
| **Spouští** | pipeline `pl_<...>` nebo ručně |
| **Vlastník** | jméno |

**Jak spustit ručně:** uprav buňku *Parameters* a dej *Run all*.
Notebook je idempotentní: opakované spuštění přepíše výstup, nevytvoří duplicity.

## 1. Parameters
Buňka je označená jako **parameter cell** (`...` → *Toggle parameter cell*).
Pipeline přepíše hodnoty přes *Base parameters* v aktivitě Notebook.
Sem patří jen hodnoty, které se mění mezi běhy nebo prostředími. Žádná ID ani tajné klíče.

In [ ]:
lakehouse_name = "lh_team_XX"   # název, ne ID - ID se dohledá za běhu
tables = ""                     # "orders,customers"; prázdné = všechny z TABLES
run_id = ""                     # z pipeline: @pipeline().RunId; prázdné = vygeneruje se
dry_run = False                 # True = jen připraví a zkontroluje data, nic nezapíše

## 2. Setup
Importy, konfigurace a prostředí. Workspace a lakehouse se zjistí za běhu,
takže stejný kód funguje v DEV, TEST i PROD.

In [ ]:
import json
import logging
import time
import uuid
from datetime import datetime, timezone

import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
import notebookutils

# --- Konfigurace (mění se jen změnou kódu, ne mezi běhy) ---
SOURCE_PREFIX = "bronze_"
TARGET_PREFIX = "silver_"
TABLES = [
    "table_a",
    "table_b",
    # ...
]
RUN_LOG_TABLE = "ops_run_log"

# --- Prostředí (zjištěno za běhu, žádná ID v kódu) ---
ctx = notebookutils.runtime.context
WORKSPACE_ID = ctx["currentWorkspaceId"]
NOTEBOOK_NAME = ctx["currentNotebookName"]
LAKEHOUSE_ID = notebookutils.lakehouse.get(lakehouse_name)["id"]
TABLES_ROOT = (f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
               f"{LAKEHOUSE_ID}/Tables/dbo")

RUN_ID = run_id or str(uuid.uuid4())
SELECTED = [t.strip().lower() for t in tables.split(",") if t.strip()] or TABLES
unknown = set(SELECTED) - set(TABLES)
if unknown:
    raise ValueError(f"Unknown tables in parameter 'tables': {sorted(unknown)}")

# --- Log do výstupu notebooku ---
logging.basicConfig(level=logging.INFO, force=True,
                    format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(NOTEBOOK_NAME)
log.info("run_id=%s lakehouse=%s tables=%s dry_run=%s",
         RUN_ID, lakehouse_name, SELECTED, dry_run)

## 3. Secrets
Tajné údaje (API klíče, hesla) **nikdy** nepiš do kódu ani do parametrů - kód je na GitHubu.
Ulož je do Azure Key Vault a načti za běhu. Pro OneLake a Fabric položky klíč nepotřebuješ,
přihlášení řeší Fabric (`getToken`). Pro databáze a API používej připojení (Connections) v pipeline.

In [ ]:
# API_KEY = notebookutils.credentials.getSecret(
#     "https://<nazev-key-vault>.vault.azure.net/", "<nazev-secretu>")

## 4. Helpers
Společné funkce: čtení a zápis Delta tabulek, technické sloupce, run log.

In [ ]:
def storage_options():
    return {
        "bearer_token": notebookutils.credentials.getToken("storage"),
        "use_fabric_endpoint": "true",
    }


def table_path(name):
    return f"{TABLES_ROOT}/{name}"


def read_table(name):
    """Čte Delta tabulku přímo z OneLake (ne přes SQL endpoint, ten se synchronizuje se zpožděním)."""
    return DeltaTable(table_path(name), storage_options=storage_options()).to_pyarrow_table()


def write_table(name, data, mode="overwrite"):
    write_deltalake(table_path(name), data, mode=mode,
                    schema_mode="overwrite" if mode == "overwrite" else None,
                    storage_options=storage_options())


def utc_now():
    return datetime.now(timezone.utc)


def add_technical_columns(data, source_table):
    """Technické sloupce Silver: začínají podtržítkem, časy v UTC."""
    n = data.num_rows
    processed_at = pa.array([utc_now()] * n, type=pa.timestamp("us", tz="UTC"))
    return (data
            .append_column("_processed_at", processed_at)
            .append_column("_run_id", pa.array([RUN_ID] * n, type=pa.string()))
            .append_column("_source_table", pa.array([source_table] * n, type=pa.string())))


RUN_LOG_SCHEMA = pa.schema([
    ("run_id", pa.string()),
    ("notebook", pa.string()),
    ("table_name", pa.string()),
    ("status", pa.string()),            # SUCCEEDED / FAILED / DRY_RUN
    ("rows_in", pa.int64()),
    ("rows_out", pa.int64()),
    ("started_at", pa.timestamp("us", tz="UTC")),
    ("finished_at", pa.timestamp("us", tz="UTC")),
    ("duration_s", pa.float64()),
    ("error_message", pa.string()),
])


def log_run(table_name, status, rows_in, rows_out, started_at, error=None):
    finished_at = utc_now()
    row = {
        "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "table_name": table_name,
        "status": status, "rows_in": rows_in, "rows_out": rows_out,
        "started_at": started_at, "finished_at": finished_at,
        "duration_s": (finished_at - started_at).total_seconds(),
        "error_message": (str(error)[:2000] if error else None),
    }
    write_table(RUN_LOG_TABLE, pa.Table.from_pylist([row], schema=RUN_LOG_SCHEMA), mode="append")
    return row

## 5. Transformace
Logika pro jednu tabulku. Pravidla čištění popiš v README (sekce Silver), ať kód a dokumentace sedí.
Při problému s daty **vyhoď chybu**, neopravuj potichu.

In [ ]:
KEYS = {
    "table_a": "table_a_id",
    "table_b": "table_b_id",
    # ...
}


def transform(name, data):
    # 1) Názvy sloupců: malá písmena, podtržítka
    data = data.rename_columns([c.lower().strip().replace(" ", "_") for c in data.column_names])

    # 2) Datové typy, Y/N -> boolean, ... (podle pravidel v README)

    # 3) Kontroly kvality: klíč nesmí chybět ani se opakovat
    key = KEYS[name]
    values = data[key].to_pylist()
    if None in values or len(set(values)) != len(values):
        raise ValueError(f"Missing or duplicate key {key}")

    return data

## 6. Run
Každá tabulka zvlášť: chyba jedné tabulky se zaloguje a ostatní pokračují.
Na konci notebook selže, pokud selhala aspoň jedna tabulka - pipeline tak chybu uvidí.

In [ ]:
results = []
for name in SELECTED:
    started_at = utc_now()
    source, target = SOURCE_PREFIX + name, TARGET_PREFIX + name
    rows_in = rows_out = None
    try:
        data = read_table(source)
        rows_in = data.num_rows
        out = add_technical_columns(transform(name, data), source)
        rows_out = out.num_rows
        if dry_run:
            status = "DRY_RUN"
        else:
            write_table(target, out)
            written = read_table(target).num_rows
            if written != rows_out:
                raise ValueError(f"Row count after write {written} != expected {rows_out}")
            status = "SUCCEEDED"
        log.info("%s: %s rows_in=%s rows_out=%s", target, status, rows_in, rows_out)
        results.append(log_run(target, status, rows_in, rows_out, started_at))
    except Exception as e:
        log.error("%s: FAILED %s", target, e)
        results.append(log_run(target, "FAILED", rows_in, rows_out, started_at, e))

## 7. Summary
Souhrn běhu. `notebookutils.notebook.exit` vrátí výsledek pipeline
(`@activity('<aktivita>').output.result.exitValue`).

In [ ]:
display(pa.Table.from_pylist(results, schema=RUN_LOG_SCHEMA).to_pandas())

failed = [r["table_name"] for r in results if r["status"] == "FAILED"]
if failed:
    raise RuntimeError(f"Failed tables: {failed}. See {RUN_LOG_TABLE} for run_id={RUN_ID}")

notebookutils.notebook.exit(json.dumps({
    "run_id": RUN_ID,
    "tables": len(results),
    "rows_out": sum(r["rows_out"] or 0 for r in results),
}))